In [58]:
import delta

#import
import ConnectionConfig as cc
from delta.tables import DeltaTable
from datetime import datetime
from pyspark.sql import SparkSession
debugging_mode=True

In [59]:
#config

cc.setupEnvironment()
spark = cc.startLocalCluster("mongodbsetup",4)
spark.getActiveSession()

Environment variables are set...


In [60]:
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")

In [61]:
#treusure tabel
df_treasure = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure.createOrReplaceTempView("treasure")

#treusure tabel
df_stage = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stage") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_stage.createOrReplaceTempView("stage")

#treusure tabel
df_treasure_stage = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure_stages") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_treasure_stage.createOrReplaceTempView("treasure_stages")

#city tabel
df_city = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "city") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_city.createOrReplaceTempView("city")

#country tabel
df_country = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "country") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_country.createOrReplaceTempView("country")


In [62]:
treasure_mongoDb = spark.sql("""
SELECT t.id,
       t.difficulty,
       t.terrain,
       t.owner_id,
         collect_list(
           named_struct(
               'city_id', c.city_id,
               'city_name', c.city_name,
               'latitude', c.latitude,
               'longitude', c.longitude,
               'postal_code', c.postal_code,
               'country_code', c.country_code
           )
       ) AS city,
        collect_list(
           named_struct(
               'code3', co.code3,
               'name', co.name
           )
       ) AS country,
       collect_list(
           named_struct(
               'stage_id', st.id,
               'container_size', st.container_size,
               'description', st.description,
               'latitude', st.latitude,
               'longitude', st.longitude,
               'sequence_number', st.sequence_number,
               'type', st.type,
               'visibility', st.visibility
           )
       ) AS stages
FROM treasure t
JOIN city c ON t.city_city_id = c.city_id
JOIN country co ON c.country_code = co.code
LEFT JOIN treasure_stages ts ON t.id = ts.treasure_id
LEFT JOIN stage st ON ts.stages_id = st.id
GROUP BY
       t.id, t.difficulty, t.terrain, t.owner_id,
       c.city_id, c.city_name, c.latitude, c.longitude, c.postal_code, c.country_code,
       co.code3, co.name
LIMIT 100
""")

treasure_mongoDb.createOrReplaceTempView("treasure_mongoDb")


spark.sql("SELECT * FROM treasure_mongoDb").show()

+--------------------+----------+-------+--------------------+--------------------+--------------------+--------------------+
|                  id|difficulty|terrain|            owner_id|                city|             country|              stages|
+--------------------+----------+-------+--------------------+--------------------+--------------------+--------------------+
|[00 03 72 3C 7C C...|         3|      2|[75 AA F7 4A 33 C...|[{[59 07 42 E8 55...|[{IND, India}, {I...|[{[1B 32 E8 9A A6...|
|[00 05 1E A2 05 8...|         0|      2|[DD AD C2 FE 29 5...|[{[38 EC 6D 3C 5A...|      [{IND, India}]|[{[6B 7D 70 BA 78...|
|[00 08 A9 BF BE 4...|         3|      2|[1C E4 7F 2B 5F 2...|[{[59 B1 A5 28 DD...|      [{IND, India}]|[{[3B 60 7F 16 34...|
|[00 08 B9 20 1B F...|         1|      2|[18 61 40 54 61 3...|[{[27 9E 9E 63 9E...|[{USA, United Sta...|[{[13 5B B8 C2 53...|
|[00 08 E2 70 CB A...|         2|      2|[FC 49 26 02 2C 3...|[{[AE A0 A3 09 22...|[{IND, India}, {I...|[{[5E 6D 6B A0

In [63]:
treasure_mongoDb.write \
    .format("json") \
    .mode("overwrite") \
    .save("treasure_export.json")